In [ ]:
import pandas as pd
import os

label_path = "data_raw/labels/"

files = [
    "train_split_Depression_AVEC2017.csv",
    "dev_split_Depression_AVEC2017.csv",
    "full_test_split.csv"
]

for file in files:
    df = pd.read_csv(os.path.join(label_path, file))
    if "PHQ8_Binary" in df.columns:
        binary_col = "PHQ8_Binary"
    else:
        binary_col = "PHQ_Binary"

    if "PHQ8_Score" in df.columns:
        score_col = "PHQ8_Score"
    else:
        score_col = "PHQ_Score"

    violation = df[(df[score_col] >= 10) & (df[binary_col] == 0)]

    if violation.empty:
        print("Nothing Wrong")
    else:
        print(f"Found {len(violation)} file(s) wrong")
        print(
            violation[["Participant_ID", score_col, binary_col]]
        )

Found 1 file(s) wrong
    Participant_ID  PHQ8_Score  PHQ8_Binary
64             409          10            0
Nothing Wrong
Nothing Wrong


In [ ]:
import pandas as pd
import os

label_path = "data_raw/labels/"
# Load Dataset 
file_path = os.path.join(label_path, 'train_split_Depression_AVEC2017.csv')
df = pd.read_csv(file_path)

# Check 409 Participant data (Before correction)
print("Before Correction:")
print(df[df['Participant_ID'] == 409][['Participant_ID', 'PHQ8_Binary', 'PHQ8_Score']])

# Change 409 Participant PHQ8_Binary value to 1 (As PHQ8_Score is 10)
df.loc[df['Participant_ID'] == 409, 'PHQ8_Binary'] = 1

# After Correction
print("\nAfter Correction:")
print(df[df['Participant_ID'] == 409][['Participant_ID', 'PHQ8_Binary', 'PHQ8_Score']])

# Save corrected dataframe as CSV file
output_file = 'train_split_Depression_AVEC2017_corrected.csv'
df.to_csv(output_file, index=False)

Before Correction:
    Participant_ID  PHQ8_Binary  PHQ8_Score
64             409            0          10

After Correction:
    Participant_ID  PHQ8_Binary  PHQ8_Score
64             409            1          10


### A data analysis script that visualizes the distribution of PHQ-8 depression questionnaire data (Figure 2.1)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os 

label_path = "data_raw/labels/"

# 1. Load file
df_detailed = pd.read_csv(os.path.join(label_path, "Detailed_PHQ8_Labels.csv"))
df_full_test_ids = pd.read_csv(os.path.join(label_path, "full_test_split.csv"))
df_dev_avec = pd.read_csv(os.path.join(label_path, "dev_split_Depression_AVEC2017.csv"))
df_train_avec = pd.read_csv(os.path.join(label_path, "train_split_Depression_AVEC2017_corrected.csv"))

# 2. Data preprocessing

# --- Prepare Test split data ---
test_data = df_detailed[df_detailed['Participant_ID'].isin(df_full_test_ids['Participant_ID'])].copy()
rename_map = {
    'PHQ_8NoInterest': 'PHQ8_NoInterest',
    'PHQ_8Depressed': 'PHQ8_Depressed',
    'PHQ_8Sleep': 'PHQ8_Sleep',
    'PHQ_8Tired': 'PHQ8_Tired',
    'PHQ_8Appetite': 'PHQ8_Appetite',
    'PHQ_8Failure': 'PHQ8_Failure',
    'PHQ_8Concentrating': 'PHQ8_Concentrating',
    'PHQ_8Moving': 'PHQ8_Moving'
}
test_data.rename(columns=rename_map, inplace=True)

# --- Definition of items and labels ---
phq_items = [
    'PHQ8_NoInterest', 'PHQ8_Depressed', 'PHQ8_Sleep', 'PHQ8_Tired', 
    'PHQ8_Appetite', 'PHQ8_Failure', 'PHQ8_Concentrating', 'PHQ8_Moving'
]

item_labels = {
    'PHQ8_NoInterest': 'Interest',
    'PHQ8_Depressed': 'Mood',
    'PHQ8_Sleep': 'Sleep',
    'PHQ8_Tired': 'Fatigue',
    'PHQ8_Appetite': 'Appetite',
    'PHQ8_Failure': 'Self-Worth',
    'PHQ8_Concentrating': 'Concentration',
    'PHQ8_Moving': 'Psychomotor'
}

# --- Data transformation helper function---
def prepare_melted_data(df, dataset_name):
    df_subset = df[phq_items].copy()
    for col in phq_items:
        df_subset[col] = pd.to_numeric(df_subset[col], errors='coerce').fillna(0).astype(int)
    
    melted = df_subset.melt(var_name='Item_Code', value_name='Score')
    melted['Item'] = melted['Item_Code'].map(item_labels)
    melted['Dataset'] = dataset_name
    return melted

# Prepare individual datasets
melted_train = prepare_melted_data(df_train_avec, "Train Split")
melted_dev = prepare_melted_data(df_dev_avec, "Dev Split")
melted_test = prepare_melted_data(test_data, "Test Split")

# Preparation of the combined dataset (All Data)
melted_combined = pd.concat([melted_train, melted_dev, melted_test])
melted_combined['Dataset'] = "All Data (n=189)"

# 3. Plotting and saving graphs (individual + combined)
datasets = [
    (melted_train, "Train Split Distribution", "phq8_distribution_train.jpg"),
    (melted_dev, "Dev Split Distribution", "phq8_distribution_dev.jpg"),
    (melted_test, "Test Split Distribution", "phq8_distribution_test.jpg"),
    (melted_combined, "Data Distribution (n=189)", "phq8_distribution_combined.jpg")
]

sns.set_style("whitegrid")

for data, title, filename in datasets:
    plt.figure(figsize=(10, 6))
    
    # Create countplot 
    sns.countplot(
        data=data, 
        x='Item', 
        hue='Score', 
        palette='viridis'
    )
    
    # Set style
    plt.title(title, fontsize=16, fontweight='bold')
    plt.ylabel("Count", fontsize=14)
    plt.xlabel("PHQ-8 Items", fontsize=14)
    plt.legend(title='Score (0-3)', loc='upper right')
    
    # Rotate x-axis labels by 45 degrees (to prevent overlap)
    plt.xticks(rotation=45, fontsize=12)
    plt.yticks(fontsize=12)
    
    plt.tight_layout()
    
    # Save file
    plt.savefig(filename, format='jpg', dpi=300)
    plt.close() 
    
    print(f"Saving complete: {filename}")

Saving complete: phq8_distribution_train.jpg
Saving complete: phq8_distribution_dev.jpg
Saving complete: phq8_distribution_test.jpg
Saving complete: phq8_distribution_combined.jpg
